# LAPD Crime Data Analysis (2020–2024)

This notebook analyzes crime incidents reported in Los Angeles between 2020 and 2024. I originally started this project using the Manitoba Rabies dataset, but after completing the initial exploration, it became clear that the dataset didn’t have enough depth or variation for meaningful data cleaning or analysis. 

The LAPD dataset provides a much stronger foundation for this course because it includes detailed information such as crime type, date and time, area, victim demographics, premises, weapon involvement, and geographic coordinates. It also contains real-world inconsistencies and missing values, which makes it more suitable for demonstrating proper data wrangling and exploratory analysis.

Because the full dataset is very large, I will be using a smaller random subset (`df_small`) for all analysis so the notebook runs efficiently while still reflecting realistic patterns in the data.


## 1. Initial Exploration

In [13]:
# Week 12: Initial dataset loading and inspection
# Purpose: Load the dataset and perform basic structural checks.

import pandas as pd

# Make sure Python can see the src/ folder
import sys
sys.path.insert(0, '..')

# Import the helpers module from src
from src import helpers

df = pd.read_csv(r"C:\Users\fpasha\OneDrive - Red River College Polytech\COMP-2040\final_project\data\Crime_Data_from_2020_to_2024.csv")

df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,211507896,04/11/2021 12:00:00 AM,11/07/2020 12:00:00 AM,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,10/21/2020 12:00:00 AM,10/18/2020 12:00:00 AM,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,12/10/2024 12:00:00 AM,10/30/2020 12:00:00 AM,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,12/24/2020 12:00:00 AM,12/24/2020 12:00:00 AM,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,10/03/2020 12:00:00 AM,09/29/2020 12:00:00 AM,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


### 1.1 Dataset Shape

In [14]:
# Just checking the size of the dataset so I know how many rows and columns I'm working with.
df.shape


(1004894, 28)

### 1.2 Column Types and Missing Values


In [15]:
# Using .info() to get a quick overview of the dataset's structure.
# This shows me the column names, their data types, and how many non-null values each column has.
# Basically, I'm checking for any obvious issues like missing data or wrong data types before I go deeper.

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1004894 entries, 0 to 1004893
Data columns (total 28 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   DR_NO           1004894 non-null  int64  
 1   Date Rptd       1004894 non-null  object 
 2   DATE OCC        1004894 non-null  object 
 3   TIME OCC        1004894 non-null  int64  
 4   AREA            1004894 non-null  int64  
 5   AREA NAME       1004894 non-null  object 
 6   Rpt Dist No     1004894 non-null  int64  
 7   Part 1-2        1004894 non-null  int64  
 8   Crm Cd          1004894 non-null  int64  
 9   Crm Cd Desc     1004894 non-null  object 
 10  Mocodes         853296 non-null   object 
 11  Vict Age        1004894 non-null  int64  
 12  Vict Sex        860263 non-null   object 
 13  Vict Descent    860251 non-null   object 
 14  Premis Cd       1004878 non-null  float64
 15  Premis Desc     1004306 non-null  object 
 16  Weapon Used Cd  327216 non-null   fl

#### 1.2.1 Missing Values Breakdown


In [16]:
# Quick breakdown of missing values in each column so I can see which parts of the dataset need cleaning later.
df.isnull().sum()


DR_NO                   0
Date Rptd               0
DATE OCC                0
TIME OCC                0
AREA                    0
AREA NAME               0
Rpt Dist No             0
Part 1-2                0
Crm Cd                  0
Crm Cd Desc             0
Mocodes            151598
Vict Age                0
Vict Sex           144631
Vict Descent       144643
Premis Cd              16
Premis Desc           588
Weapon Used Cd     677678
Weapon Desc        677678
Status                  1
Status Desc             0
Crm Cd 1               11
Crm Cd 2           935740
Crm Cd 3          1002580
Crm Cd 4          1004830
LOCATION                0
Cross Street       850666
LAT                     0
LON                     0
dtype: int64

### 1.3 Descriptive Statistics


In [17]:
# Quick summary stats for the numeric columns so I can spot weird ranges or outliers.
df.describe()


,DR_NO,TIME OCC,AREA,Rpt Dist No,Part 1-2,Crm Cd,Vict Age,Premis Cd,Weapon Used Cd,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LAT,LON
count,1.004894e+06,1.004894e+06,1.004894e+06,1.004894e+06,1.004894e+06,1.004894e+06,1.004894e+06,1.004878e+06,327216.000000,1.004883e+06,69154.000000,2314.000000,64.00000,1.004894e+06,1.004894e+06
mean,2.202190e+08,1.339901e+03,1.069153e+01,1.115613e+03,1.400313e+00,5.001386e+02,2.891862e+01,3.056108e+02,363.950797,4.998992e+02,958.097796,984.015990,991.21875,3.399820e+01,-1.180908e+02
std,1.319070e+07,6.510761e+02,6.110248e+00,6.111596e+02,4.899620e-01,2.052638e+02,2.199267e+01,2.192913e+02,123.737665,2.050643e+02,110.358509,52.350982,27.06985,1.610789e+00,5.582654e+00
min,8.170000e+02,1.000000e+00,1.000000e+00,1.010000e+02,1.000000e+00,1.100000e+02,-4.000000e+00,1.010000e+02,101.000000,1.100000e+02,210.000000,310.000000,821.00000,0.000000e+00,-1.186676e+02
25%,2.106168e+08,9.000000e+02,5.000000e+00,5.870000e+02,1.000000e+00,3.310000e+02,0.000000e+00,1.010000e+02,311.000000,3.310000e+02,998.000000,998.000000,998.00000,3.401470e+01,-1.184305e+02
50%,2.209158e+08,1.420000e+03,1.100000e+01,1.139000e+03,1.000000e+00,4.420000e+02,3.000000e+01,2.030000e+02,400.000000,4.420000e+02,998.000000,998.000000,998.00000,3.405890e+01,-1.183225e+02
75%,2.311102e+08,1.900000e+03,1.600000e+01,1.613000e+03,2.000000e+00,6.260000e+02,4.400000e+01,5.010000e+02,400.000000,6.260000e+02,998.000000,998.000000,998.00000,3.416490e+01,-1.182739e+02
max,2.521040e+08,2.359000e+03,2.100000e+01,2.199000e+03,2.000000e+00,9.560000e+02,1.200000e+02,9.760000e+02,516.000000,9.560000e+02,999.000000,999.000000,999.00000,3.433430e+01,0.000000e+00


### 1.4 Initial Observations

- The dataset is **huge** with a little over **1 million rows**, so I need to be aware of performance when I start doing heavier operations.
- A few columns are basically useless. **Crm Cd 2**, **Crm Cd 3**, and **Crm Cd 4** are almost completely empty, so they probably won’t help with anything.
- The weapon fields have a ton of missing values. **Weapon Used Cd** and **Weapon Desc** are missing around **677k rows**, which tells me most incidents either didn’t involve a weapon or the info wasn’t recorded.
- **Vict Sex** and **Vict Descent** both have around **144k missing values**, which is a lot. I’ll need to figure out how to handle that later since those fields matter for analysis.
- **Cross Street** is missing more than **850k rows**, so the street level detail is weak. I’ll probably rely more on the `LAT` and `LON` columns instead.
- I noticed some weird values in the numeric summary. **Vict Age** goes down to `-4`, which obviously isn’t valid and will need cleaning.
- **LON** has a max value of `0`, which doesn’t make sense for LA. That looks like a placeholder or a bad entry that I’ll need to fix.


### 1.5 Analytical Questions

- What types of crimes are the most common in this dataset, and how do they compare when I group them by the `Crm Cd Desc` column?
- How does crime activity change by time of day? I want to look at patterns using the `TIME OCC` column to see when incidents happen the most.
- Are certain areas reporting more incidents than others? I want to compare crime counts across different `AREA NAME` values to see which parts of the city stand out.


## 2. Data Cleaning

### 2.1 Fix invalid ages
The `Vict Age` column has some values that are negative. That obviously isn’t possible, so these rows would mess up any age‑based analysis. I’m treating any age below 0 as missing because it keeps the dataset clean and avoids pulling wrong conclusions later.


In [18]:
# Some rows have negative ages like -4, which isn't realistic.
# These values would mess up any analysis involving victim age,
# so I'm replacing any age below 0 with NaN to mark them as missing.
df['Vict Age'] = df['Vict Age'].apply(lambda x: x if x >= 0 else None)


### 2.2 Remove invalid longitude values
The `LON` column has some values that are `0`, which doesn’t make sense for Los Angeles. Real LA coordinates should be around -118, so a longitude of 0 basically means the location wasn’t recorded properly. These rows would mess up any mapping or location analysis, so I’m removing them now to keep the dataset clean.


In [19]:
# Removing rows where LON is 0 because that isn't a real coordinate for LA.
# Keeping these rows would cause problems later when I try to map or analyze
# crime locations, since a longitude of 0 would place the point somewhere
# in the Atlantic Ocean. Dropping them now keeps the dataset consistent.
df = df[df['LON'] != 0]


### 2.3 Drop columns that are basically empty
The columns `Crm Cd 2`, `Crm Cd 3`, and `Crm Cd 4` are almost completely empty. They barely contain any real data, so keeping them won’t add anything useful to the analysis. Dropping them now keeps the dataset cleaner and makes it easier to focus on the columns that actually matter.


In [20]:
# Dropping Crm Cd 2, 3, and 4 because they are almost entirely empty.
# These columns don't provide any meaningful information and would just
# clutter the dataset, so removing them now keeps things cleaner and easier
# to work with for the rest of the project.
df = df.drop(columns=['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4'])


### 2.4 Convert date columns to datetime
The `Date Rptd` and `DATE OCC` columns are stored as strings right now. Converting them to proper datetime objects makes it easier to work with dates later, especially if I want to look at trends over time. Using `errors='coerce'` also helps because any bad or weird date formats will automatically turn into `NaT` instead of breaking the code.


In [21]:
# Converting the date columns from strings into proper datetime objects.
# This makes it easier to filter by date, extract things like year or month,
# and run any time-based analysis later on. Using errors='coerce' makes sure
# that any invalid or messy date entries get turned into NaT instead of causing errors.
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce')
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')


C:\Users\fpasha\AppData\Local\Temp\ipykernel_16428\279976682.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce')
C:\Users\fpasha\AppData\Local\Temp\ipykernel_16428\279976682.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')


**Note about the date parsing warning:**  
Pandas showed a warning saying it couldn't infer the exact date format and had to fall back to a slower parser. The important part is that the conversion still worked because the dates in this dataset are already in a standard `YYYY-MM-DD` format. Since pandas successfully converted both `Date Rptd` and `DATE OCC` into proper datetime values, I don’t need to specify a manual format or change anything here.


### 2.5 Handle missing values
A few columns in this dataset have missing values, but not all of them need to be filled. Some fields like `Cross Street` or `Weapon Used Cd` are naturally incomplete because not every crime report includes that information. For this project, I’m only fixing the columns where the missing values are small and easy to handle. The goal here is to clean up obvious gaps without changing the meaning of the data.


In [22]:
# Handling missing values in a few key columns.
# Some columns naturally have missing data (like Cross Street or Weapon Used Cd),
# so I'm leaving those as-is. But for columns where the missing values are small
# and easy to fix, I'm filling them with a simple placeholder to keep things consistent.

# Filling missing Premis Cd with -1 to mark "unknown" without dropping rows.
df['Premis Cd'] = df['Premis Cd'].fillna(-1)

# Filling missing Status values with "UNKNOWN" so the column stays usable.
df['Status'] = df['Status'].fillna('UNKNOWN')


**Note:** I used `-1` for missing values in `Premis Cd` because it’s a numeric code column, and using a number keeps the column consistent without dropping any rows. For `Status`, I used `"UNKNOWN"` since that column is text-based, and filling it with a string keeps it readable and easy to work with. Both choices keep the dataset clean without changing the meaning of the data.


## 3. Helper Module

### 3.1 Create helper module
To keep the notebook organized, I’m creating a separate Python file called `helpers.py`. This file will store small reusable functions that I can import and use throughout the analysis. For now, I’m adding a simple test function to confirm that the module loads correctly.


In [23]:

helpers.hello()


'helpers.py is working'

### 3.2 Add basic helper functions
Now that the helper module is created, I’m adding a few small reusable functions to keep the notebook cleaner. These functions handle common tasks like filtering by year, counting values in a column, and getting the top N categories. Each function includes a clear docstring explaining what it does, what it takes in, and what it returns.


### 3.3 Test helper functions
Now that the helper functions are added, I’m importing the module and running a few quick tests to make sure everything works as expected. This confirms that the functions load properly and return the kind of output I need for the analysis section.


In [24]:

# Testing filter_by_year to make sure it returns only rows from the year I specify.
# If this works, I should see a DataFrame preview with DATE OCC values from 2020 only.
helpers.filter_by_year(df, 2020).head()


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,LOCATION,Cross Street,LAT,LON
0,211507896,2021-04-11,2020-11-07,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354.0,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020-10-21,2020-10-18,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230.0,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024-12-10,2020-10-30,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354.0,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020-12-24,2020-12-24,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,STREET,NaN,NaN,IC,Invest Cont,331.0,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020-10-03,2020-09-29,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,ALLEY,NaN,NaN,IC,Invest Cont,420.0,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


In [25]:
# Testing count_by_column to confirm it produces a clean frequency table.
# This helps me quickly check which crime descriptions appear the most.
helpers.count_by_column(df, 'Crm Cd Desc').head()

Crm Cd Desc
VEHICLE - STOLEN                                           115127
BATTERY - SIMPLE ASSAULT                                    74465
BURGLARY FROM VEHICLE                                       63460
THEFT OF IDENTITY                                           62511
VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)     61028
Name: count, dtype: int64

In [26]:
# Testing top_n to verify it returns the top N most common values in a column.
# This is useful later when I summarize which areas have the most incidents.
helpers.top_n(df, 'AREA NAME')

AREA NAME
Central        69474
77th Street    61623
Pacific        59340
Southwest      57369
Hollywood      52107
N Hollywood    51014
Olympic        49940
Southeast      49788
Newton         49075
Wilshire       48106
Name: count, dtype: int64

### 3.4 Add one more helper function
To support the analysis section, I added a small grouping helper that counts how many rows fall into each category of a column. This will be useful later when summarizing incidents by area, crime type, or year. It keeps the notebook cleaner by avoiding repeated groupby logic.


In [27]:
# Testing group_and_count to confirm it returns a sorted summary table.
helpers.group_and_count(df, 'AREA NAME').head()


AREA NAME
Central        69474
77th Street    61623
Pacific        59340
Southwest      57369
Hollywood      52107
dtype: int64

### 3.5 Summary of the Helper Module

To keep the notebook clean and focused on the analysis, I moved repeated logic into a separate helper module located in `src/helpers.py`. This module now contains reusable functions for filtering by year, counting values in a column, getting the top N categories, and grouping data to produce quick summaries. Each function includes a clear docstring explaining what it does, its parameters, and what it returns. Using a helper module makes the workflow easier to maintain and keeps the analysis section readable by hiding low‑level operations that don’t need to appear in the main notebook.


## 4. Data Cleaning

### 4.1 Identify Cleaning Tasks
Before cleaning the dataset, I reviewed the results from Section 2 to identify the specific issues that need to be fixed. Based on the exploration, the main cleaning tasks are:

- Convert date columns to proper datetime format.
- Remove rows where the date could not be parsed.
- Extract useful time-based features such as year, month, and hour.
- Standardize text columns (strip whitespace, fix casing, handle inconsistent labels).
- Handle invalid ages (e.g., 0 or extremely high values).
- Remove or handle rows with missing or invalid coordinates.
- Drop columns that are not useful for analysis or have too many missing values.
- Ensure all categorical fields are consistent and ready for grouping/visualization.

These tasks will prepare the dataset for the analysis and visualizations in Week 14.


In [28]:
# 4.2 Data Cleaning - Step 1: Clean and extract date information

# Make a copy so we don't modify the original df accidentally
clean_df = df.copy()

# Convert the date column to datetime
clean_df['DATE OCC'] = pd.to_datetime(clean_df['DATE OCC'], errors='coerce')

# Drop rows where the date could not be parsed
clean_df = clean_df.dropna(subset=['DATE OCC'])

# Extract useful time features
clean_df['YEAR'] = clean_df['DATE OCC'].dt.year
clean_df['MONTH'] = clean_df['DATE OCC'].dt.month
clean_df['HOUR'] = clean_df['DATE OCC'].dt.hour

clean_df.head()


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,LOCATION,Cross Street,LAT,LON,YEAR,MONTH,HOUR
0,211507896,2021-04-11,2020-11-07,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,7800 BEEMAN AV,NaN,34.2124,-118.4092,2020,11,0
1,201516622,2020-10-21,2020-10-18,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,ATOLL AV,N GAULT,34.1993,-118.4203,2020,10,0
2,240913563,2024-12-10,2020-10-30,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,14600 SYLVAN ST,NaN,34.1847,-118.4509,2020,10,0
3,210704711,2020-12-24,2020-12-24,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,6000 COMEY AV,NaN,34.0339,-118.3747,2020,12,0
4,201418201,2020-10-03,2020-09-29,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,4700 LA VILLA MARINA,NaN,33.9813,-118.4350,2020,9,0


### Cleaning Step 1: Date Parsing and Feature Extraction
I converted the DATE OCC column into a proper datetime format and removed rows where the date could not be parsed. Then I extracted the YEAR, MONTH, and HOUR fields, which will be useful for time‑based analysis later in the project.


### 4.2 Clean Invalid Ages
The `Vict Age` column contains unrealistic values such as 0, negative numbers, or ages above 100. These values would distort any age‑based analysis, so they need to be handled before moving forward. My goal here is to make the age field reliable without removing too much data.


In [30]:
# 4.2 Clean Invalid Ages

# Convert Vict Age to numeric in case it was read as a string.
# Using errors='coerce' turns anything non-numeric into NaN, which is easier to handle.
clean_df['Vict Age'] = pd.to_numeric(clean_df['Vict Age'], errors='coerce')

# Identify unrealistic ages.
# Ages <= 0 or > 100 are not meaningful for analysis and should be treated as missing.
invalid_age_mask = (clean_df['Vict Age'] <= 0) | (clean_df['Vict Age'] > 100)

# Replace invalid ages with NaN so they don't influence averages or distributions.
clean_df.loc[invalid_age_mask, 'Vict Age'] = pd.NA

# -----------------------------
# VERIFICATION: PROVE IT WORKED
# -----------------------------

# 1. Check if any invalid ages remain after cleaning.
remaining_invalid = clean_df[(clean_df['Vict Age'] <= 0) | (clean_df['Vict Age'] > 100)]
print("Remaining invalid ages (should be empty):")
display(remaining_invalid)

# 2. Quick summary to confirm the column looks reasonable.
clean_df['Vict Age'].describe()


Remaining invalid ages (should be empty):


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,LOCATION,Cross Street,LAT,LON,YEAR,MONTH,HOUR


count    733886.000000
mean         39.504285
std          15.571505
min           2.000000
25%          28.000000
50%          37.000000
75%          50.000000
max          99.000000
Name: Vict Age, dtype: float64

### Cleaning Step 2: Why I Cleaned Ages This Way
I chose to convert the `Vict Age` column to numeric and replace unrealistic values with `NaN` because incorrect ages would distort any age‑related patterns in the analysis. Values like 0, negative ages, or extremely high ages are not meaningful in this context, so treating them as missing is the safest option. I also added a verification step to confirm that no invalid ages remain after cleaning. This approach keeps the dataset realistic without removing entire rows, which helps preserve as much information as possible for the later analysis.
